# 1. MetaQ: a small public count example

Run this notebook from the GitHub repository root with the **MetaQ-env** kernel. Install and patch MetaQ as in the repository README before starting. Download the MetacellFormer-evaluation dataset and set `METACELLFORMER_EVALUATION` to its directory, or edit the default below.

This uses the first 100 cells of the released K562 **unnormalized UMI count** matrix. It demonstrates membership generation and raw-count aggregation. The CPU example runs MetaQ for its recorded 300 epochs. Output goes to `example_outputs/`.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

repo = Path.cwd().resolve()
assert (repo / "scripts/prepare_metaq.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
source = data_root / "inputs/perturbation/K562.h5ad"
assert source.is_file(), f"Download the evaluation dataset: {source}"
work = repo / "example_outputs/metaq"
work.mkdir(parents=True, exist_ok=True)
raw_file = work / "K562_100_cells.h5ad"

backed = ad.read_h5ad(source, backed="r")
try:
    raw = backed[:100].to_memory()
finally:
    backed.file.close()
counts = sparse.csr_matrix(raw.X)
assert raw.n_obs == 100 and raw.obs_names.is_unique
assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
assert np.allclose(counts.data, np.rint(counts.data))
raw.X = counts
raw.write_h5ad(raw_file)
print(raw.shape)

In [ ]:
env = os.environ.copy()
env.update(METAQ_NUM_WORKERS="0", METAQ_PIN_MEMORY="0", OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
subprocess.run([sys.executable, str(repo / "scripts/prepare_metaq.py"),
                "--input", str(raw_file), "--output-dir", str(work / "run"),
                "--device", "cpu", "--batch-size", "16"],
               cwd=repo, env=env, check=True)
subprocess.run([sys.executable, str(repo / "scripts/aggregate_metacells.py"),
                str(raw_file), str(work / "run/memberships.tsv"),
                str(work / "metacells.h5ad"), "--source-id", "K562_example"],
               cwd=repo, env=env, check=True)

In [ ]:
membership = pd.read_csv(work / "run/memberships.tsv", sep="\t", dtype=str)
metacells = ad.read_h5ad(work / "metacells.h5ad")
assert membership.source_cell_id.tolist() == raw.obs_names.tolist()
assert int(metacells.obs.n_cells.sum()) == raw.n_obs
assert metacells.var_names.equals(raw.var_names)
for group in metacells.obs_names:
    selected = membership.metacell_id.to_numpy() == group
    expected = np.asarray(counts[selected].sum(axis=0)).ravel()
    np.testing.assert_array_equal(metacells[group].X.toarray().ravel(), expected)
print("Verified", len(metacells), "metacells from", len(raw), "source cells")